# EVDetect — Bootstrap Confidence Intervals for Model Comparison

This notebook computes **percentile bootstrap 95% confidence intervals** (1,000 resamples) for the Accuracy, Precision, Recall, F1-macro scores of the five ensemble-based candidate models:

- Random Forest
- Random Forest + SMOTE
- XGBoost
- XGBoost + SMOTE
- Stacking Classifier

across all four experimental scenarios:

1. Binary — Full dataset (both states)
2. Binary — Charging state only
3. Multiclass — Full dataset (both states)
4. Multiclass — Charging state only

Confidence intervals are computed on the **held-out test set predictions** of the already-trained, saved models (produced by notebooks `02_multiclass.ipynb` and `03_binary.ipynb`). No retraining occurs here.

**Requires:**
- `EVSE-B-PowerCombined.csv` in a local `data/` folder
- trained model artifacts in a local `models/` folder


In [4]:
# --- Path configuration ---
MODELS_DIR = 'models/'

N_BOOTSTRAPS = 1000
RANDOM_STATE = 42

In [6]:
# Import the necessary libraries
import numpy as np
import pandas as pd
import joblib
import os
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

## Bootstrap CI function

Percentile method: resample the test set with replacement, recompute the macro average F1, Precision, Recall and Accuracy,  on each resample, and take the 2.5th / 97.5th percentiles of the resulting distribution. No normality assumption is made.

In [9]:
def bootstrap_metrics_ci(y_true, y_pred, n_boot=N_BOOTSTRAPS, seed=RANDOM_STATE):
    """
    Computes 95% Percentile Bootstrap Confidence Intervals for:
    - Accuracy
    - Precision (macro)
    - Recall (macro)
    - F1 (macro)
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    # Initialize empty arrays
    scores = {
        'accuracy': np.empty(n_boot),
        'precision': np.empty(n_boot),
        'recall': np.empty(n_boot),
        'f1': np.empty(n_boot)
    }

    # Point estimates
    point_estimates = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0)
    }

    # Perform bootstrap resampling
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        y_true_b = y_true[idx]
        y_pred_b = y_pred[idx]

        scores['accuracy'][b] = accuracy_score(y_true_b, y_pred_b)
        scores['precision'][b] = precision_score(y_true_b, y_pred_b, average='macro', zero_division=0)
        scores['recall'][b] = recall_score(y_true_b, y_pred_b, average='macro', zero_division=0)
        scores['f1'][b] = f1_score(y_true_b, y_pred_b, average='macro', zero_division=0)

    # Package results
    results = {}
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        results[metric] = {
            'point': point_estimates[metric],
            'ci_lower': np.percentile(scores[metric], 2.5),
            'ci_upper': np.percentile(scores[metric], 97.5)
        }
    return results

## Model loading

In [14]:
scenario_models = {
    'Binary — Full dataset': {
        'Random Forest':          joblib.load(os.path.join(MODELS_DIR, 'random_forest_bin_both_best_model.pkl')),
        'Random Forest + SMOTE':  joblib.load(os.path.join(MODELS_DIR, 'random_forest_bin_both_smote_best_model.pkl')),
        'XGBoost':                joblib.load(os.path.join(MODELS_DIR, 'xgboost_bin_both_best_model.pkl')),
        'XGBoost + SMOTE':        joblib.load(os.path.join(MODELS_DIR, 'xgboost_bin_both_smote_best_model.pkl')),
        'Stacking Classifier':    joblib.load(os.path.join(MODELS_DIR, 'stacking_clf_bin_both_best_model.pkl')),
    },
    'Binary — Charging only': {
        'Random Forest':          joblib.load(os.path.join(MODELS_DIR, 'random_forest_bin_charging_best_model.pkl')),
        'Random Forest + SMOTE':  joblib.load(os.path.join(MODELS_DIR, 'random_forest_bin_charging_smote_best_model.pkl')),
        'XGBoost':                joblib.load(os.path.join(MODELS_DIR, 'xgboost_bin_charging_best_model.pkl')),
        'XGBoost + SMOTE':        joblib.load(os.path.join(MODELS_DIR, 'xgboost_bin_charging_smote_best_model.pkl')),
        'Stacking Classifier':    joblib.load(os.path.join(MODELS_DIR, 'stacking_clf_bin_charging_best_model.pkl')),
    },
    'Multiclass — Full dataset': {
        'Random Forest':          joblib.load(os.path.join(MODELS_DIR, 'random_forest_multi_both_best_model.pkl')),
        'Random Forest + SMOTE':  joblib.load(os.path.join(MODELS_DIR, 'random_forest_multi_both_smote_best_model.pkl')),
        'XGBoost':                joblib.load(os.path.join(MODELS_DIR, 'xgboost_multi_both_best_model.pkl')),
        'XGBoost + SMOTE':        joblib.load(os.path.join(MODELS_DIR, 'xgboost_multi_both_smote_best_model.pkl')),
        'Stacking Classifier':    joblib.load(os.path.join(MODELS_DIR, 'stacking_clf_multi_both_best_model.pkl')),
    },
    'Multiclass — Charging only': {
        'Random Forest':          joblib.load(os.path.join(MODELS_DIR, 'random_forest_multi_charging_best_model.pkl')),
        'Random Forest + SMOTE':  joblib.load(os.path.join(MODELS_DIR, 'random_forest_multi_charging_smote_best_model.pkl')),
        'XGBoost':                joblib.load(os.path.join(MODELS_DIR, 'xgboost_multi_charging_best_model.pkl')),
        'XGBoost + SMOTE':        joblib.load(os.path.join(MODELS_DIR, 'xgboost_multi_charging_smote_best_model.pkl')),
        'Stacking Classifier':    joblib.load(os.path.join(MODELS_DIR, 'stacking_clf_multi_charging_best_model.pkl')),
    },
}

C:\Users\klaza\anaconda3\envs\basic_python\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\klaza\anaconda3\envs\basic_python\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\klaza\anaconda3\envs\basic_python\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from ver

## Data Loading

In [87]:
# 1. Load Data (Load BOTH string and encoded versions for Multiclass)
X_test_bin_both = joblib.load("models/TestSetsBinary/X_test.pkl")
Y_test_bin_both = joblib.load("models/TestSetsBinary/Y_test.pkl")

X_test_bin_charging = joblib.load("models/TestSetsBinaryCharging/X_test.pkl")
Y_test_bin_charging = joblib.load("models/TestSetsBinaryCharging/Y_test.pkl")

X_test_multi_both = joblib.load("models/TestSetsMulti/X_test.pkl")
Y_test_multi_both_str = joblib.load("models/TestSetsMulti/Y_test.pkl")
Y_test_multi_both_enc = joblib.load("models/TestSetsMulti/Y_test_encoded.pkl")

X_test_multi_charging = joblib.load("models/TestSetsMultiCharging/X_test.pkl")
Y_test_multi_charging_str = joblib.load("models/TestSetsMultiCharging/Y_test.pkl")
Y_test_multi_charging_enc = joblib.load("models/TestSetsMultiCharging/Y_test_encoded.pkl")

# Map them out (Binary only needs one Y_test)
scenario_data = {
    'Binary — Full dataset':      (X_test_bin_both, Y_test_bin_both, None),
    'Binary — Charging only':     (X_test_bin_charging, Y_test_bin_charging, None),
    'Multiclass — Full dataset':  (X_test_multi_both, Y_test_multi_both_str, Y_test_multi_both_enc),
    'Multiclass — Charging only': (X_test_multi_charging, Y_test_multi_charging_str, Y_test_multi_charging_enc),
}

In [88]:
# 2. The Loop

rows = []

for scenario, models_dict in scenario_models.items():
    X_test_s, Y_test_str, Y_test_enc = scenario_data[scenario]
    
    for model_name, model in models_dict.items():
        # Get hard predictions only (NO probabilities needed, drastically speeds up execution)
        y_pred = model.predict(X_test_s)
        
        # DYNAMIC MATCHING: Is the model predicting numbers (XGBoost) or strings (Random Forest)?
        is_numeric_pred = isinstance(y_pred[0], (int, float, np.integer, np.floating))
        
        if Y_test_enc is not None and is_numeric_pred:
            current_y_true = Y_test_enc
        else:
            current_y_true = Y_test_str
            
        # Run bootstrap on the 4 fast metrics
        metrics_results = bootstrap_metrics_ci(current_y_true, y_pred)
        
        # Store results
        row = {
            'Scenario': scenario,
            'Model': model_name
        }
        
        for metric, vals in metrics_results.items():
            row[f'{metric.capitalize()}'] = round(vals['point'], 4)
            row[f'{metric.capitalize()} CI Lower'] = round(vals['ci_lower'], 4)
            row[f'{metric.capitalize()} CI Upper'] = round(vals['ci_upper'], 4)
            
        rows.append(row)
        print(f"{scenario:<28} {model_name:<24} | F1: {metrics_results['f1']['point']:.4f} | Acc: {metrics_results['accuracy']['point']:.4f}")

df_cis = pd.DataFrame(rows)
df_cis.to_csv('bootstrap_fast_metrics.csv', index=False)
print("\nSaved fast bootstrap metrics to bootstrap_fast_metrics.csv!")

Binary — Full dataset        Random Forest            | F1: 0.8727 | Acc: 0.9454
Binary — Full dataset        Random Forest + SMOTE    | F1: 0.8630 | Acc: 0.9365
Binary — Full dataset        XGBoost                  | F1: 0.8774 | Acc: 0.9475
Binary — Full dataset        XGBoost + SMOTE          | F1: 0.8348 | Acc: 0.9130
Binary — Full dataset        Stacking Classifier      | F1: 0.8739 | Acc: 0.9481
Binary — Charging only       Random Forest            | F1: 0.8096 | Acc: 0.8618
Binary — Charging only       Random Forest + SMOTE    | F1: 0.8073 | Acc: 0.8596
Binary — Charging only       XGBoost                  | F1: 0.8103 | Acc: 0.8731
Binary — Charging only       XGBoost + SMOTE          | F1: 0.8104 | Acc: 0.8542
Binary — Charging only       Stacking Classifier      | F1: 0.8130 | Acc: 0.8586
Multiclass — Full dataset    Random Forest            | F1: 0.6516 | Acc: 0.6311
Multiclass — Full dataset    Random Forest + SMOTE    | F1: 0.6457 | Acc: 0.6287
Multiclass — Full dataset   

In [92]:
# Full results table
df_cis

,Scenario,Model,Accuracy,Accuracy CI Lower,Accuracy CI Upper,Precision,Precision CI Lower,Precision CI Upper,Recall,Recall CI Lower,Recall CI Upper,F1,F1 CI Lower,F1 CI Upper
0,Binary — Full dataset,Random Forest,0.9454,0.9424,0.9485,0.8796,0.8717,0.8868,0.8661,0.8586,0.8742,0.8727,0.8661,0.8793
1,Binary — Full dataset,Random Forest + SMOTE,0.9365,0.9331,0.9396,0.8437,0.8355,0.8512,0.8861,0.8795,0.8929,0.8630,0.8561,0.8697
2,Binary — Full dataset,XGBoost,0.9475,0.9447,0.9503,0.8848,0.8774,0.8922,0.8705,0.8632,0.8782,0.8774,0.8710,0.8837
3,Binary — Full dataset,XGBoost + SMOTE,0.9130,0.9093,0.9165,0.7927,0.7854,0.7996,0.9088,0.9031,0.9141,0.8348,0.8281,0.8410
4,Binary — Full dataset,Stacking Classifier,0.9481,0.9452,0.9512,0.8990,0.8911,0.9065,0.8528,0.8449,0.8610,0.8739,0.8668,0.8807
5,Binary — Charging only,Random Forest,0.8618,0.8530,0.8704,0.8065,0.7938,0.8189,0.8129,0.8003,0.8258,0.8096,0.7976,0.8217
6,Binary — Charging only,Random Forest + SMOTE,0.8596,0.8510,0.8684,0.8032,0.7911,0.8153,0.8117,0.7992,0.8248,0.8073,0.7958,0.8188
7,Binary — Charging only,XGBoost,0.8731,0.8643,0.8817,0.8389,0.8265,0.8514,0.7903,0.7770,0.8035,0.8103,0.7978,0.8225
8,Binary — Charging only,XGBoost + SMOTE,0.8542,0.8459,0.8626,0.7946,0.7830,0.8059,0.8341,0.8228,0.8454,0.8104,0.7989,0.8217
9,Binary — Charging only,Stacking Classifier,0.8586,0.8505,0.8667,0.8000,0.7886,0.8117,0.8305,0.8189,0.8421,0.8130,0.8024,0.8240


## Manuscript-ready formatting

Compact `value [lower–upper]` strings, pivoted with scenarios as columns — matching the layout of the supplementary table in the paper.

In [101]:
df_fmt = df_cis.copy()

# Note the updated column names: 'F1 CI Lower' and 'F1 CI Upper'
df_fmt['F1-macro [95% CI]'] = df_fmt.apply(
    lambda r: f"{r['F1']:.4f} [{r['F1 CI Lower']:.4f}–{r['F1 CI Upper']:.4f}]", axis = 1
)

pivot = df_fmt.pivot(index = 'Model', columns = 'Scenario', values = 'F1-macro [95% CI]')
pivot

Scenario,Binary — Charging only,Binary — Full dataset,Multiclass — Charging only,Multiclass — Full dataset
Model,,,,
Random Forest,0.8096 [0.7976–0.8217],0.8727 [0.8661–0.8793],0.8397 [0.8297–0.8491],0.6516 [0.6457–0.6572]
Random Forest + SMOTE,0.8073 [0.7958–0.8188],0.8630 [0.8561–0.8697],0.8400 [0.8298–0.8494],0.6457 [0.6400–0.6515]
Stacking Classifier,0.8130 [0.8024–0.8240],0.8739 [0.8668–0.8807],0.8439 [0.8340–0.8528],0.6587 [0.6529–0.6646]
XGBoost,0.8103 [0.7978–0.8225],0.8774 [0.8710–0.8837],0.8422 [0.8319–0.8514],0.6551 [0.6493–0.6606]
XGBoost + SMOTE,0.8104 [0.7989–0.8217],0.8348 [0.8281–0.8410],0.8462 [0.8365–0.8546],0.6308 [0.6251–0.6364]


In [103]:
# Format the dataframe columns into "Value [Lower-Upper]" style
df_fmt = df_cis.copy()

metrics_to_format = ['Accuracy', 'Precision', 'Recall', 'F1']

for m in metrics_to_format:
    df_fmt[f'{m} [95% CI]'] = df_fmt.apply(
        lambda r: f"{r[m]:.3f} [{r[f'{m} CI Lower']:.3f}–{r[f'{m} CI Upper']:.3f}]", axis=1
    )

# Print LaTeX tables manually (Bypassing Jinja2 completely!)
for m in metrics_to_format:
    pivot_table = df_fmt.pivot(index='Model', columns='Scenario', values=f'{m} [95% CI]')
    
    print(f"\n%%% --- LaTeX Table for {m} --- %%%")
    print("\\begin{table}[h!]")
    print("\\centering")
    
    # Setup column formats (Left align for Model, Center for data columns)
    col_format = "l" + "c" * len(pivot_table.columns)
    print(f"\\begin{{tabular}}{{{col_format}}}")
    print("\\toprule")
    
    # Headers
    headers = ["Model"] + list(pivot_table.columns)
    print(" & ".join(headers) + " \\\\")
    print("\\midrule")
    
    # Rows
    for index, row in pivot_table.iterrows():
        row_vals = [str(index)] + [str(val) for val in row.values]
        print(" & ".join(row_vals) + " \\\\")
        
    print("\\bottomrule")
    print("\\end{tabular}")
    print(f"\\caption{{Bootstrap 95\\% Confidence Intervals for {m}}}")
    print(f"\\label{{tab:ci_{m.lower()}}}")
    print("\\end{table}")


%%% --- LaTeX Table for Accuracy --- %%%
\begin{table}[h!]
\centering
\begin{tabular}{lcccc}
\toprule
Model & Binary — Charging only & Binary — Full dataset & Multiclass — Charging only & Multiclass — Full dataset \\
\midrule
Random Forest & 0.862 [0.853–0.870] & 0.945 [0.942–0.949] & 0.849 [0.839–0.859] & 0.631 [0.625–0.637] \\
Random Forest + SMOTE & 0.860 [0.851–0.868] & 0.936 [0.933–0.940] & 0.850 [0.840–0.858] & 0.629 [0.622–0.635] \\
Stacking Classifier & 0.859 [0.851–0.867] & 0.948 [0.945–0.951] & 0.854 [0.844–0.862] & 0.643 [0.637–0.650] \\
XGBoost & 0.873 [0.864–0.882] & 0.948 [0.945–0.950] & 0.857 [0.848–0.866] & 0.644 [0.638–0.650] \\
XGBoost + SMOTE & 0.854 [0.846–0.863] & 0.913 [0.909–0.916] & 0.851 [0.841–0.860] & 0.632 [0.626–0.639] \\
\bottomrule
\end{tabular}
\caption{Bootstrap 95\% Confidence Intervals for Accuracy}
\label{tab:ci_accuracy}
\end{table}

%%% --- LaTeX Table for Precision --- %%%
\begin{table}[h!]
\centering
\begin{tabular}{lcccc}
\toprule
Model & Binar